In [ ]:
import os

from jmap_backup.tiny_jmap import TinyJMAPClient

In [ ]:
token = "..."
username = "..."

In [ ]:
client = TinyJMAPClient(
    hostname=os.environ.get("JMAP_HOSTNAME", "api.fastmail.com"),
    username=username,
    token=token,
)

In [ ]:
account_id = client.get_account_id()


In [ ]:
inbox_res = client.make_jmap_call(
    {
        "using": ["urn:ietf:params:jmap:core", "urn:ietf:params:jmap:mail"],
        "methodCalls": [
            [ 
                "Mailbox/get", {
                    "accountId": account_id,
                    "ids": None
                }, 
                "0" 
            ]
        ]
    }
)
inbox_res

In [ ]:
email_res = client.make_jmap_call(
    {
        "using": ["urn:ietf:params:jmap:core", "urn:ietf:params:jmap:mail"],
        "methodCalls": [
            [ 
                "Email/query", {
                    "accountId": account_id,
                    "filter": {
                        #"inMailbox": "P-F"
                        "inMailbox": "P6-" # Archive
                    },
                    "sort": [{
                        "property": "receivedAt",
                        "isAscending": False
                    }],
                    "position": 0,
                    "collapseThreads": False,
                    "limit": 10,
                    "calculateTotal": True
                }, 
                "0" 
            ],
            # Then we fetch the threadId of each of those messages
            [ 
                "Email/get", {
                    "accountId": account_id,
                    "#ids": {
                        "name": "Email/query",
                        "path": "/ids",
                        "resultOf": "0"
                    },
                    "properties": [ "threadId", "from", "to", "subject" ]
                }, 
                "1"
            ]
        ]
    }
)
email_res

In [ ]:
client.session

In [ ]:
email_text_res = client.make_jmap_call(
    {
        "using": ["urn:ietf:params:jmap:core", "urn:ietf:params:jmap:mail"],
        "methodCalls": [
            [ 
                "Email/get", {
                    "accountId": account_id,
                    "ids": ["StthjenFfGuZ"],
                    "properties": [ "bodyStructure", "bodyValues", "threadId", "subject", "from", "receivedAt" ],
                    "bodyProperties": [ "partId", "blobId", "name", "headers", "type"],
                    "fetchAllBodyValues": True,
                }, 
                "1"
            ]
        ]
    }
)
import json
print(json.dumps(email_text_res, indent=2))

https://www.fastmailusercontent.com/jmap/download/u8ece405f/G470ba097fa126e7f903342dd4c6c167b5e00e085/?type=text/html

In [ ]:
from email import generator
def write_eml_file(msg, filename="emails/testmail.eml"):
    with open(filename, 'w') as file:
        emlGenerator = generator.Generator(file)
        emlGenerator.flatten(msg)

In [ ]:
from email.message import EmailMessage
from email.headerregistry import Address
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.nonmultipart import MIMENonMultipart

from urllib.request import Request, urlopen
from urllib.parse import quote_plus
from base64 import b64encode

from textwrap import wrap
import re

subject = email_text_res['methodResponses'][0][1]['list'][0]['subject']
# to_name = email_text_res['methodResponses'][0][1]['list'][0]['to'][0]['name']
# to_email = email_text_res['methodResponses'][0][1]['list'][0]['to'][0]['email']
# to = Address(display_name=to_name, addr_spec=to_email)
# from_name = email_text_res['methodResponses'][0][1]['list'][0]['from'][0]['name']
from_email = email_text_res['methodResponses'][0][1]['list'][0]['from'][0]['email']
# from_ = Address(display_name=from_name or '', addr_spec=from_email)


# html_body = email_text_res['methodResponses'][0][1]['list'][0]['bodyValues']['1.2']['value']
# text_body = email_text_res['methodResponses'][0][1]['list'][0]['bodyValues']['1.1']['value']
account_id = client.get_account_id()

def download_blob(blob_id, name, full_type, client):
    account_id = client.get_account_id()
    download_url = client.session["downloadUrl"] \
        .replace("{accountId}", account_id)      \
        .replace("{blobId}", blob_id)            \
        .replace("{name}", quote_plus(name))                 \
        .replace("{type}", full_type)
    print(f"Downloading attachment from {download_url}")
    r = Request(
        download_url, 
        headers={
            "Authorization": f"Bearer {client.token}",
        },)
    f = urlopen(r)
    data = f.read()
    return "\n".join(wrap(str(b64encode(data), 'utf-8'), 76))

def build_message_part(sub_part, email_text_res):
    print("called")
    full_type = sub_part['type']
    main_type = full_type.split('/')[0]
    subtype = full_type.split('/')[1]
    headers = {h['name']: h['value'] for h in sub_part.get('headers', [])}
    if main_type == 'multipart':
        msg_part = MIMEMultipart(subtype if subtype else 'mixed')
        for part in sub_part['subParts']:
            msg_part.attach(build_message_part(part, email_text_res))
    elif main_type == 'text':
        part_id = sub_part['partId']
        charset = None
        if 'Content-Type' in headers:
            # Try to extract charset from Content-Type header
            match = re.search(r'charset=([^\s;]+)', headers['Content-Type'], re.IGNORECASE)
            if match:
                charset = match.group(1).strip('"').strip("'")
        msg_part = MIMEText(
            email_text_res['methodResponses'][0][1]['list'][0]['bodyValues'][part_id]['value'], _subtype=subtype, _charset=charset)
    else:
        part_id = sub_part['partId']
        blob_id = sub_part['blobId']
        name = sub_part['name']
        msg_part = MIMENonMultipart(main_type, subtype)
        msg_part.set_payload(download_blob(blob_id, name, full_type, client))
    
    for header in sub_part.get('headers', []):
        if header['name'] == 'Content-Type':
            print(f"Content-Type header: {header['value']}")
            continue
        msg_part.add_header(header['name'], header['value'].replace('\r', '').replace('\n', ''))
    print(msg_part["Content-Type"])
    print("exited")
    return msg_part

body_structure = email_text_res['methodResponses'][0][1]['list'][0]['bodyStructure']

msg = build_message_part(body_structure, email_text_res)

write_eml_file(msg, f"emails/{from_email.split('@')[0]}-{subject}.eml")


## Exploring the Email/changes method to get incremental changes

In [ ]:
email_res = client.make_jmap_call(
    {
        "using": ["urn:ietf:params:jmap:core", "urn:ietf:params:jmap:mail"],
        "methodCalls": [
            [ 
                "Email/query", {
                    "accountId": account_id,
                    "filter": {
                        "inMailbox": "P-Gxh" # Hotmail
                    },
                    "collapseThreads": True,
                    "position": 1,
                    "limit": 1,
                    "calculateTotal": True
                }, 
                "0" 
            ],
            # Then we fetch the threadId of each of those messages
            [ 
                "Email/get", {
                    "accountId": account_id,
                    "#ids": {
                        "name": "Email/query",
                        "path": "/ids",
                        "resultOf": "0"
                    },
                    "properties": [ "bodyStructure", "bodyValues", "threadId", "subject", "from"],
                    "bodyProperties": [ "partId", "blobId", "name", "headers", "type"],
                    "fetchAllBodyValues": True,
                }, 
                "1"
            ]
        ]
    }
)
email_res

In [ ]:
email_changes = client.make_jmap_call(
    {
        "using": ["urn:ietf:params:jmap:core", "urn:ietf:params:jmap:mail"],
        "methodCalls": [
            [ 
                "Email/queryChanges", {
                    "accountId": account_id,
                    "filter": {
                        "inMailbox": "P-F"
                    },
                    "sort": [{
                        "property": "receivedAt",
                        "isAscending": True
                    }],
                    "sinceQueryState": "J217742:0",
                    "collapseThreads": False,
                    "maxChanges": 10,
                    "calculateTotal": True
                }, 
                "0" 
            ],
            # Then we fetch the threadId of each of those messages
            [ 
                "Email/get", {
                    "accountId": account_id,
                    "#ids": {
                        "name": "Email/queryChanges",
                        "path": "/added/*/id",
                        "resultOf": "0"
                    },
                    "properties": [ "bodyStructure", "bodyValues", "threadId", "subject", "from"],
                    "bodyProperties": [ "partId", "blobId", "name", "headers", "type"],
                    "fetchAllBodyValues": True,
                }, 
                "1"
            ]
        ]
    }
)
email_changes